# Spatial-cutoff violations — all saved alignment experiments
**Read-only evaluation of saved alignments. No alignment or recovery is rerun. No ground-truth plan is evaluated.**

Includes 18 DLPFC pairs, 10 MERFISH pairs, 5 simulated-overlap conditions, 4 cropped-window tests, and 14 Deep-STARmap pairs (51 total). Separate bar plots are produced for these five groups; windows are kept separate from the simulated overlap sweep.

For positive-support edges $e=(i,j),f=(k,l)$, compute $\Delta_{ef}=|d_A(i,k)-d_B(j,l)|$ in **μm**. Report the exact fraction of distinct unordered edge pairs with $\Delta_{ef}>\epsilon$, and the exact mass-weighted fraction $V_\epsilon=\sum_{e,f}p_ep_f\mathbf1\{\Delta_{ef}>\epsilon\}/(\sum_ep_e)^2$. Self-pairs contribute zero to the latter numerator. Shared-source/target pairs are included. All positive weights are retained; no sampling or argmax is used. Numerical tolerance is $10^{-9}$ μm.

The **anchor pairwise cutoff** is the primary test. The **recovery cutoff** is a supplementary numerical reference: recovery uses anchor-distance residuals and is not guaranteed to enforce this final pairwise bound. Small windows have no recovery stage and are marked N/A.

Coordinates and cutoffs are converted using each saved decision's calibration. **Deep-STARmap uses the historical 1 input unit = 1 μm convention recorded in the alignment pipeline; its physical XY calibration has not been independently verified.** Its plots explicitly mark this qualification. These results describe saved matrices/source versions, not an unexecuted newer algorithm.

In [ ]:
import os
os.environ.setdefault('NUMBA_CACHE_DIR', '/private/tmp/raftup_violation_numba')
os.environ.setdefault('MPLCONFIGDIR', '/private/tmp/raftup_violation_mpl')
from pathlib import Path
import json, hashlib, time
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
from scipy.spatial.distance import cdist
from IPython.display import display
ROOT = Path('/Users/salovjade/Library/CloudStorage/Dropbox/raftup_revision_828')
REPRO = Path('/Users/salovjade/Library/CloudStorage/Dropbox/RAFTUP_after0320/raftup_reproduce')
OUT = ROOT / 'all_exp/distortion_violation_audit/results_all'
OUT.mkdir(parents=True, exist_ok=True)
BLOCK = 512  # memory-bounded exact enumeration; changing this does not change definitions
ATOL = 1e-9

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1024*1024), b''): h.update(b)
    return h.hexdigest()

def read_dlpfc(section):
    folder = REPRO / '1_DLPFC_all_exp/DLPFC12' / section
    a = sc.read_visium(folder, count_file=section+'_filtered_feature_bc_matrix.h5')
    a.var_names_make_unique()
    gt = pd.read_csv(folder/'gt/tissue_positions_list_GTs.txt', header=None, index_col=0)
    a.obs['original_clusters'] = gt[6]
    return a[a.obs.dropna().index].copy()  # identical filtering/order to saved alignment loader

from scipy import sparse
import matplotlib
matplotlib.use('Agg')
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import Image
from IPython.display import Image
import gc


In [ ]:
WINDOWS = {
 'dlpfc_same_top':('151675-151675_topwindow/overlapping_subsliceA.h5ad','151675-151675_topwindow/overlapping_subsliceB.h5ad'),
 'dlpfc_same_bottom':('151675-151675_bottomwindow/simple_overlapping_subsliceA.h5ad','151675-151675_bottomwindow/simple_overlapping_subsliceB.h5ad'),
 'dlpfc_cross_7576':('151675-151676_window/test_1_subsliceA.h5ad','151675-151676_window/test_1_subsliceB.h5ad'),
 'merfish_1419_window':('0.14-0.19_window/test_4_subsliceA.h5ad','0.14-0.19_window/test_4_subsliceB.h5ad')}
cases=[]
for group, dirname, expected in [('DLPFC','dlpfc',18),('MERFISH','merfish',10),('test','test',9),('Deep-STARmap','deep_starmap',14)]:
    folders=sorted(f.parent for f in (ROOT/'all_exp'/dirname).glob('*/alignment_matrix*.npy'))
    assert len(folders)==expected, (group,len(folders),expected)
    for folder in folders:
        decision_path=folder/('decision.json' if group=='DLPFC' else 'cutoff_decision.json')
        d=json.loads(decision_path.read_text()); units=d['spatial_units']
        conversion=float(units['internal_units_per_um'])
        if group=='DLPFC':
            pars=d['final']; anchor=pars['ds_spatial_um']; recovery=pars['rf_spatial_um']
        elif group=='test':
            pars=d['workflow_decision']
            if 'config' in pars: anchor=pars['config']['spatial_down']; recovery=pars['config']['spatial_full']
            else: anchor=pars['ds_spatial']; recovery=pars['rf_spatial']
            anchor/=conversion
            if recovery is not None: recovery/=conversion
        else:
            anchor=d['ds_spatial']/conversion; recovery=d['rf_spatial']/conversion
        actual_group=('Simulated overlap' if folder.name.startswith('fig9_') else 'Cropped windows') if group=='test' else group
        # Prefer sparse Deep-STARmap exports; other dense outputs are memory mapped.
        path=folder/'alignment_matrix_sparse.npz'
        if not path.exists(): path=next(folder.glob('alignment_matrix*.npy'))
        cases.append(dict(dataset=actual_group,case=folder.name,folder=folder,path=path,
            decision_path=decision_path,decision=d,
            scale_a=units['micrometers_per_input_unit_A'],scale_b=units['micrometers_per_input_unit_B'],
            anchor_cutoff_um=float(anchor),recovery_cutoff_um=None if recovery is None else float(recovery)))
assert len(cases)==51
display(pd.DataFrame([{k:c[k] for k in ['dataset','case','anchor_cutoff_um','recovery_cutoff_um']} for c in cases]))

def coords(c):
    name=c['case']; group=c['dataset']
    if group=='DLPFC':
        a,b=read_dlpfc('1515'+name[:2] if int(name[:2])<60 else '1516'+name[:2]),read_dlpfc('1515'+name[2:] if int(name[2:])<60 else '1516'+name[2:])
    elif group=='MERFISH':
        cache=ROOT/'cutoff_automation_test/merfish_automation/slice_cache'
        a,b=[ad.read_h5ad(cache/f'slice_-0.{s}.h5ad') for s in (name[:2],name[2:])]
    elif group=='Simulated overlap':
        folder=ROOT/'1_simulated_data_exp/151673_simulatedData';pct=int(name.split('_')[-1])
        a=ad.read_h5ad(folder/'DLPFC_151673original_spotIndex.h5')
        pseudo='1.0' if pct==100 else '1'
        b=ad.read_h5ad(folder/f'DLPFC_151673_overlap={pct}%_pseudocount_{pseudo}_spotIndex.h5')
    elif group=='Cropped windows':
        a,b=[ad.read_h5ad(REPRO/'3_synthetic_window'/f) for f in WINDOWS[name]]
    else:
        cfg=json.loads((c['folder']/'run_config.json').read_text())
        a,b=[ad.read_h5ad(cfg[k]['path']) for k in ['A','B']]
        order=json.loads((c['folder']/'cell_order.json').read_text())
        assert a.obs_names.tolist()==order['A'] and b.obs_names.tolist()==order['B']
    xa=np.asarray(a.obsm['spatial'],float)[:,:2]*c['scale_a']
    xb=np.asarray(b.obsm['spatial'],float)[:,:2]*c['scale_b']
    hashes=[hashlib.sha256('\n'.join(o.obs_names).encode()).hexdigest() for o in [a,b]]
    return xa,xb,hashes


In [ ]:
def support(path):
    if path.suffix=='.npz':
        p=sparse.load_npz(path).tocsr();p.sum_duplicates();p.eliminate_zeros()
        assert np.isfinite(p.data).all() and (p.data>=0).all()
        q=p.tocoo();return q.row,q.col,q.data.astype(float),p.shape
    p=np.load(path,mmap_mode='r');ii=[];jj=[];ww=[]
    for start in range(0,p.shape[0],512):
        block=np.asarray(p[start:start+512]);assert np.isfinite(block).all() and (block>=0).all()
        i,j=np.nonzero(block>0);ii.append(i+start);jj.append(j);ww.append(block[i,j])
    return np.concatenate(ii),np.concatenate(jj),np.concatenate(ww).astype(float),p.shape

def audit(ii,jj,w,shape,xa,xb,cutoffs):
    assert shape==(len(xa),len(xb)) and np.isfinite(xa).all() and np.isfinite(xb).all()
    m=len(w); assert m>1 and w.sum()>0
    # Restrict distance tables to coordinates participating in the saved support.
    ui,ir=np.unique(ii,return_inverse=True);uj,jr=np.unique(jj,return_inverse=True)
    da,db=cdist(xa[ui],xa[ui]),cdist(xb[uj],xb[uj])
    counts={k:0 for k in cutoffs}; masses={k:0. for k in cutoffs};maximum=0.;seen=0
    for start in range(0,m,BLOCK):
        u=np.arange(start,min(start+BLOCK,m))
        for other in range(start,m,BLOCK):
            v=np.arange(other,min(other+BLOCK,m))
            delta=np.abs(da[ir[u,None],ir[v][None,:]]-db[jr[u,None],jr[v][None,:]])
            weights=w[u,None]*w[v][None,:]
            if other==start:
                mask=np.triu(np.ones(delta.shape,dtype=bool),k=1);delta,weights=delta[mask],weights[mask]
            else:delta,weights=delta.ravel(),weights.ravel()
            seen+=delta.size
            if delta.size:maximum=max(maximum,float(delta.max()))
            for k,cutoff in cutoffs.items():
                bad=delta>cutoff+ATOL;counts[k]+=int(bad.sum());masses[k]+=float(weights[bad].sum())
    assert seen==m*(m-1)//2
    return [dict(threshold_role=k,cutoff_um=float(t),support_edges=m,evaluated_distinct_edge_pairs=seen,
        violating_edge_pairs=counts[k],support_violation_rate=counts[k]/seen,
        mass_weighted_violation_rate=2*masses[k]/float(w.sum()**2),maximum_distortion_um=maximum,
        total_transported_mass=float(w.sum()),coverage_A=len(ui)/shape[0],coverage_B=len(uj)/shape[1]) for k,t in cutoffs.items()]

# Known-answer and unit-invariance checks.
x=np.array([[0.,0.],[1.,0.]]); ids=np.arange(2);w=np.ones(2)
r=audit(ids,ids,w,(2,2),x,3*x,{'test':1.})[0]
assert r['support_violation_rate']==1 and r['mass_weighted_violation_rate']==.5
assert audit(ids,ids,w,(2,2),x,x+7,{'test':.1})[0]['support_violation_rate']==0
assert audit(ids,ids,w,(2,2),x/1.38,3*x/1.38,{'test':1/1.38})[0]['mass_weighted_violation_rate']==.5


In [ ]:
rows=[]; provenance=[]
for c in cases:
    start=time.perf_counter();xa,xb,order_hashes=coords(c)
    ii,jj,w,shape=support(c['path'])
    cuts={'anchor_pairwise':c['anchor_cutoff_um']}
    if c['recovery_cutoff_um'] is not None:cuts['recovery_numeric_reference']=c['recovery_cutoff_um']
    result=audit(ii,jj,w,shape,xa,xb,cuts)
    note='historical unit convention; physical XY calibration unverified' if c['dataset']=='Deep-STARmap' else 'saved spatial calibration'
    rows.extend(dict(dataset=c['dataset'],case=c['case'],unit='μm',calibration_note=note,**r) for r in result)
    provenance.append(dict(dataset=c['dataset'],case=c['case'],path=str(c['path']),sha256=sha256(c['path']),
        decision=c['decision'],observation_order_sha256=order_hashes,coordinate_scale_A=c['scale_a'],
        coordinate_scale_B=c['scale_b'],seconds=time.perf_counter()-start,
        saved_source_fingerprints=json.loads((c['folder']/'source_fingerprints.json').read_text()) if (c['folder']/'source_fingerprints.json').exists() else None))
    pd.DataFrame(rows).to_csv(OUT/'violation_summary.csv',index=False)
    print(c['dataset'],c['case'],'completed:',round(time.perf_counter()-start,1),'s',flush=True)
    del xa,xb,ii,jj,w;gc.collect()
results=pd.DataFrame(rows)
assert results[['dataset','case']].drop_duplicates().shape[0]==51
(OUT/'provenance.json').write_text(json.dumps(dict(cases=provenance,enumeration='exact',support_threshold=0,comparison_atol_um=ATOL),indent=2))
display(results.style.format({'support_violation_rate':'{:.3%}','mass_weighted_violation_rate':'{:.3%}',
    'coverage_A':'{:.2%}','coverage_B':'{:.2%}','cutoff_um':'{:.2f}','maximum_distortion_um':'{:.2f}'}))


In [ ]:
# One figure per dataset, with primary and supplementary thresholds in separate panels.
for dataset, frame in results.groupby('dataset',sort=False):
    names=frame['case'].drop_duplicates().tolist()
    if dataset=='Simulated overlap':names=sorted(names,key=lambda s:int(s.split('_')[-1]))
    fig,axes=plt.subplots(2,1,figsize=(max(10,len(names)*.62),8),sharex=True,constrained_layout=True)
    for ax,role,title in zip(axes,['anchor_pairwise','recovery_numeric_reference'],
                             ['Anchor pairwise cutoff (primary)','Recovery cutoff (numerical reference only)']):
        sub=frame[frame.threshold_role==role].set_index('case').reindex(names)
        xx=np.arange(len(names));width=.38
        for offset,col,color,label in [(-width/2,'support_violation_rate','#4679A6','Support-pair violation rate'),
                                      (width/2,'mass_weighted_violation_rate','#D98642','Mass-weighted violation rate')]:
            bars=ax.bar(xx+offset,sub[col].to_numpy()*100,width,color=color,label=label)
            ax.bar_label(bars,fmt='%.2f',fontsize=7,padding=2)
        high=max(1.,float(sub[['support_violation_rate','mass_weighted_violation_rate']].max().max()*100) if sub.cutoff_um.notna().any() else 1.)
        ax.set_ylim(0,high*1.60);ax.set_ylabel('Violations (%)');ax.set_title(title);ax.grid(axis='y',alpha=.2);ax.set_axisbelow(True)
        for j,name in enumerate(names):
            t=sub.loc[name,'cutoff_um']
            ax.text(j,high*1.22,(f'ε={t:g} μm' if pd.notna(t) else 'N/A'),ha='center',fontsize=7)
        ax.legend(fontsize=8,loc='upper left',bbox_to_anchor=(0,1.02))
    labels=[s.replace('section_','').replace('__','–').replace('fig9_overlap_','Overlap ') for s in names]
    axes[-1].set_xticks(np.arange(len(names)),labels,rotation=45,ha='right')
    suffix='\nμm under historical convention; physical XY calibration unverified' if dataset=='Deep-STARmap' else ''
    fig.suptitle(dataset+' — final-alignment spatial violations'+suffix,fontsize=11)
    stem=dataset.lower().replace(' ','_').replace('-','_')
    fig.savefig(OUT/f'{stem}_violation_bars.png',dpi=200)
    fig.savefig(OUT/f'{stem}_violation_bars.pdf')
    display(Image(filename=str(OUT/f'{stem}_violation_bars.png')));plt.close(fig)
print('Tables and figures:',OUT)


## Interpretation
A nonzero primary rate shows that the saved final plan does not strictly inherit the anchor pairwise cutoff. The recovery panel applies a different numeric threshold to the same pairwise statistic; it does not test recovery's anchor-residual feasibility. Zero violations on these data do not establish a general guarantee. Bilateral coverage and support- versus mass-weighted rates help interpret tiny weights and rejection. Source versions are those associated with saved results; no claim is made that these outputs were recomputed using a newer max-clique solver. For datasets lacking a saved order manifest, the original input loader/order is reproduced and matrix dimensions checked. Deep-STARmap order is checked against its saved cell manifest.

In [ ]:
# Final full-resolution alignments only, evaluated at each experiment's full cutoff.
# Equal weight per experiment; ordinary support-pair violation rate (not mass-weighted).
# No anchor matrices are loaded. Cropped windows use their direct full-point alignment cutoff.
# Full recovery cutoff is an anchor-residual parameter; this is a numerical pairwise audit.
# Deep-STARmap's saved μm calibration is a historical convention, not independently verified.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display, Image
summary_dir=Path('/Users/salovjade/Library/CloudStorage/Dropbox/raftup_revision_828/all_exp/distortion_violation_audit/results_all')
source=pd.read_csv(summary_dir/'violation_summary.csv',dtype={'case':str})
full=source[source.threshold_role.eq('recovery_numeric_reference')].copy()
# Windows are aligned directly at full resolution; their saved ds_spatial is the
# actual cutoff used for all points, even though no separate recovery is run.
windows=source[source.dataset.eq('Cropped windows') & source.threshold_role.eq('anchor_pairwise')].copy()
windows['threshold_role']='direct_full_alignment'
full=pd.concat([full,windows],ignore_index=True)
full.to_csv(summary_dir/'full_alignment_violation_rates.csv',index=False)
assert not full.duplicated(['dataset','case']).any()
order=['DLPFC','MERFISH','Simulated overlap','Cropped windows','Deep-STARmap']
means=full.groupby('dataset')['support_violation_rate'].agg(mean_violation_rate='mean',n_experiments='size').reindex(order)
means['n_experiments']=means['n_experiments'].fillna(0).astype(int)
means['mean_violation_percent']=100*means.mean_violation_rate
means['status']=np.where(means.n_experiments>0,'available','missing result')
assert means.n_experiments.sum()==51 and means.mean_violation_rate.notna().all()
means.to_csv(summary_dir/'dataset_mean_ordinary_violation.csv')
fig,ax=plt.subplots(figsize=(9,5.2),constrained_layout=True)
values=means.mean_violation_percent.to_numpy()
bars=ax.bar(np.arange(len(order)),values,width=.62,color='#8DD3C7')
ax.bar_label(bars,labels=[f'{v:.2f}%' if np.isfinite(v) else '' for v in values],padding=5)
for j,v in enumerate(values):
    if not np.isfinite(v):ax.text(j,.12,'N/A',ha='center',va='bottom',color='#C74343')
labels=[f'{name}\n(n={int(means.loc[name,"n_experiments"])})' if means.loc[name,'n_experiments'] else name for name in order]
ax.set_xticks(np.arange(len(order)),labels,fontsize=9)
ax.set_ylabel('Mean Violation Rate (%)')
ax.set_title('Spatial Cutoff Violation Rates for dataset with n pairs')
ax.set_ylim(0,max(1.,np.nanmax(values)*1.23))
ax.grid(axis='y',alpha=.2);ax.set_axisbelow(True)
fig.savefig(summary_dir/'dataset_mean_ordinary_violation.png',dpi=200)
fig.savefig(summary_dir/'dataset_mean_ordinary_violation.pdf')
display(Image(filename=str(summary_dir/'dataset_mean_ordinary_violation.png')))
plt.close(fig)


## DLPFC: how much do violating distances differ?
Each plot compares the within-A distance $d_A(i,k)$ with the within-B distance $d_B(j,l)$ for two retained correspondences. The band $|d_A-d_B|\leq\epsilon_{full}$ is the admissible region for this pairwise audit. Colored points are violating combinations; color gives the **absolute distance difference** $\Delta=|d_A-d_B|$ in μm. The **excess** is $\Delta-\epsilon_{full}$. The inset shows the excess distribution over **all** violating combinations (1-μm bins). These are distance-space plots, not tissue-coordinate plots.

Statistics and histograms enumerate all distinct positive-support edge pairs exactly. Only scatter points are sampled, independently with a fixed seed; target counts are 12,000 violating and 2,000 nonviolating pairs per plot. Consequently scatter density must not be used to estimate prevalence. Extreme violations are also retained separately in a CSV with observation indices. No alignment is rerun.

In [ ]:
# Requires setup, case discovery, and support() cells above; no all-dataset recomputation required.
import heapq
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
DETAIL = OUT/'dlpfc_distance_diagnostics'
DETAIL.mkdir(exist_ok=True)
old_summary=pd.read_csv(OUT/'violation_summary.csv',dtype={'case':str})
stats=[]
for c in [c for c in cases if c['dataset']=='DLPFC']:
    xa,xb,_=coords(c);ii,jj,w,shape=support(c['path'])
    assert shape==(len(xa),len(xb))
    cutoff=float(c['recovery_cutoff_um']);m=len(w)
    ui,ir=np.unique(ii,return_inverse=True);uj,jr=np.unique(jj,return_inverse=True)
    da,db=cdist(xa[ui],xa[ui]),cdist(xb[uj],xb[uj])
    previous=old_summary[(old_summary['case']==c['case']) & old_summary.threshold_role.eq('recovery_numeric_reference')].iloc[0]
    expected_bad=int(previous.violating_edge_pairs);total=m*(m-1)//2
    probabilities={True:min(1.,12000/max(expected_bad,1)),False:min(1.,2000/max(total-expected_bad,1))}
    rng=np.random.default_rng(20260922+int(c['case']))
    # Difference cannot exceed the largest within-slice distance.
    bins=np.arange(0,np.ceil(max(da.max(),db.max()))+2,1.)
    hist=np.zeros(len(bins)-1,dtype=np.int64)
    scatter=[];top=[];count=0;sum_delta=0.;maximum=0.;seen=0
    for start in range(0,m,BLOCK):
        u=np.arange(start,min(start+BLOCK,m))
        for other in range(start,m,BLOCK):
            v=np.arange(other,min(other+BLOCK,m))
            aa=da[ir[u,None],ir[v][None,:]];bb=db[jr[u,None],jr[v][None,:]]
            if other==start:
                row,col=np.triu_indices(len(u),1)
            else:
                row=np.repeat(np.arange(len(u)),len(v));col=np.tile(np.arange(len(v)),len(u))
            a,b=aa[row,col],bb[row,col];delta=np.abs(a-b);bad=delta>cutoff+ATOL;seen+=len(delta)
            indices=np.flatnonzero(bad);values=delta[indices];count+=len(indices)
            if len(values):
                sum_delta+=float(values.sum());maximum=max(maximum,float(values.max()))
                hist+=np.histogram(values-cutoff,bins=bins)[0]
                # Exact ten largest deviations, including their original observation indices.
                selected=indices[np.argpartition(values,-min(10,len(values)))[-min(10,len(values)):]]
                for q in selected:
                    e,f=int(u[row[q]]),int(v[col[q]])
                    entry=(float(delta[q]),e,f,float(a[q]),float(b[q]))
                    if len(top)<10:heapq.heappush(top,entry)
                    elif entry>top[0]:heapq.heapreplace(top,entry)
            for flag in [False,True]:
                candidates=np.flatnonzero(bad if flag else ~bad)
                size=rng.binomial(len(candidates),probabilities[flag])
                if size:
                    sel=rng.choice(candidates,size=size,replace=False)
                    scatter.append(np.column_stack([a[sel],b[sel],delta[sel],np.full(size,flag)]))
    assert seen==total and count==expected_bad and hist.sum()==count
    points=np.vstack(scatter) if scatter else np.empty((0,4))
    df=pd.DataFrame(points,columns=['distance_A_um','distance_B_um','difference_um','violates'])
    df['excess_um']=df.difference_um-cutoff;df.to_csv(DETAIL/f'{c["case"]}_scatter_sample.csv',index=False)
    pd.DataFrame({'excess_bin_left_um':bins[:-1],'excess_bin_right_um':bins[1:],'count':hist}).to_csv(DETAIL/f'{c["case"]}_excess_histogram.csv',index=False)
    extremes=[dict(difference_um=d,excess_um=d-cutoff,distance_A_um=a,distance_B_um=b,
                   source_i=int(ii[e]),target_j=int(jj[e]),source_k=int(ii[f]),target_l=int(jj[f]))
              for d,e,f,a,b in sorted(top,reverse=True)]
    pd.DataFrame(extremes).to_csv(DETAIL/f'{c["case"]}_largest_violations.csv',index=False)
    mean=sum_delta/count if count else np.nan
    stats.append(dict(pair=c['case'],cutoff_um=cutoff,violating_combinations=count,total_combinations=total,
        ordinary_violation_rate=count/total,mean_difference_violating_um=mean,
        max_difference_violating_um=maximum if count else np.nan,
        mean_excess_um=mean-cutoff,max_excess_um=maximum-cutoff if count else np.nan))
    fig,ax=plt.subplots(figsize=(9,7),constrained_layout=True)
    limit=max(da.max(),db.max())*1.03;t=np.array([0,limit])
    ax.fill_between(t,np.maximum(0,t-cutoff),np.minimum(limit,t+cutoff),color='#E8EEEE',zorder=0)
    ax.plot(t,t,color='#718080',lw=.8)
    ax.plot(t,t+cutoff,'--',color='#607676',lw=1);ax.plot(t,t-cutoff,'--',color='#607676',lw=1)
    good=df[df.violates==0];badpoints=df[df.violates==1]
    ax.scatter(good.distance_A_um,good.distance_B_um,s=3,c='#BCC8C8',alpha=.35,rasterized=True)
    norm=Normalize(vmin=cutoff,vmax=max(cutoff+1,maximum))
    artist=ax.scatter(badpoints.distance_A_um,badpoints.distance_B_um,c=badpoints.difference_um,
                      norm=norm,cmap='viridis',s=5,alpha=.65,rasterized=True)
    fig.colorbar(artist,ax=ax,pad=.02,label='Distance difference |dA − dB| (μm)')
    ax.set(xlim=(0,limit),ylim=(0,limit),xlabel='Within-slice distance dA(i, k) (μm)',
           ylabel='Within-slice distance dB(j, l) (μm)',
           title=f'DLPFC {c["case"]} · full cutoff = {cutoff:g} μm\nViolation rate: {count/total:.2%} · mean difference: {mean:.1f} μm · maximum: {maximum:.1f} μm')
    ax.set_aspect('equal',adjustable='box')
    inset=ax.inset_axes([.08,.62,.36,.28])
    nonzero=np.flatnonzero(hist)
    if len(nonzero):
        end=nonzero[-1]+1
        inset.stairs(hist[:end]/count*100,bins[:end+1],fill=True,color='#84A59D')
    inset.set_xlabel('Excess above cutoff (μm)',fontsize=8)
    inset.set_ylabel('Violating pairs (%)',fontsize=8)
    inset.tick_params(labelsize=7);inset.set_title('All violations · 1-μm bins',fontsize=8)
    fig.savefig(DETAIL/f'{c["case"]}_distance_violation.png',dpi=180)
    fig.savefig(DETAIL/f'{c["case"]}_distance_violation.pdf')
    display(Image(filename=str(DETAIL/f'{c["case"]}_distance_violation.png')));plt.close(fig)
    print(c['case'],'completed',flush=True)
    del da,db,xa,xb;gc.collect()
distance_summary=pd.DataFrame(stats)
distance_summary.to_csv(DETAIL/'dlpfc_violation_distance_summary.csv',index=False)
display(distance_summary)


In [ ]:
# Average of the 18 DLPFC inset distributions: equal weight per pair.
# Normalize each pair's histogram by its own number of violating combinations,
# then average aligned 1-μm bins. This is conditional on violation, not a violation rate.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display, Image
excess_dir=Path('/Users/salovjade/Library/CloudStorage/Dropbox/raftup_revision_828/all_exp/distortion_violation_audit/results_all/dlpfc_distance_diagnostics')
summary=pd.read_csv(excess_dir/'dlpfc_violation_distance_summary.csv',dtype={'pair':str})
assert len(summary)==18
histograms=[]
for row in summary.itertuples():
    h=pd.read_csv(excess_dir/f'{row.pair}_excess_histogram.csv')
    assert h['count'].sum()==row.violating_combinations
    assert np.allclose(h.excess_bin_right_um-h.excess_bin_left_um,1.)
    if row.violating_combinations>0:
        histograms.append(h.set_index('excess_bin_left_um')['count']/row.violating_combinations)
assert histograms, 'No violations to summarize.'
mean_probability=pd.concat(histograms,axis=1).fillna(0).mean(axis=1).sort_index()
assert np.isclose(mean_probability.sum(),1.)
# Trim empty trailing bins, retaining the complete observed tail.
last=mean_probability[mean_probability>0].index.max()
mean_probability=mean_probability.loc[:last]
left=mean_probability.index.to_numpy(dtype=float)
assert np.allclose(left,np.arange(len(left)))
mean_hist=pd.DataFrame({'excess_bin_left_um':left,'excess_bin_right_um':left+1,
    'mean_violating_pairs_percent':100*mean_probability.to_numpy()})
mean_hist.to_csv(excess_dir/'dlpfc_mean_excess_histogram.csv',index=False)
# Exact mean from per-pair sums, rather than approximate histogram bin centers.
mean_excess=summary.loc[summary.violating_combinations>0,'mean_excess_um'].mean()
fig,ax=plt.subplots(figsize=(8,4.8),constrained_layout=True)
ax.stairs(mean_hist.mean_violating_pairs_percent,np.r_[left,left[-1]+1],fill=True,color='#8DD3C7')
ax.axvline(mean_excess,color='#436B68',linestyle='--',linewidth=1.4,
           label=f'Mean excess = {mean_excess:.2f} μm')
ax.set_xlabel('Excess above full cutoff (μm)')
ax.set_ylabel('Mean fraction of violating pairs (%)')
ax.set_title(f'DLPFC: mean excess distribution (n={len(histograms)})')
ax.set_xlim(0,left[-1]+1);ax.set_ylim(bottom=0)
overall_rate=summary['ordinary_violation_rate'].mean()
ax.text(.02,.30,f'Total violations\n{overall_rate:.2%}',transform=ax.transAxes,fontsize=11,fontweight='bold',color='#254B46',bbox=dict(facecolor='white',alpha=.8,edgecolor='none',pad=4))
ax.legend(frameon=False);ax.grid(axis='y',alpha=.2);ax.set_axisbelow(True)
fig.savefig(excess_dir/'dlpfc_mean_excess_distribution.png',dpi=200)
fig.savefig(excess_dir/'dlpfc_mean_excess_distribution.pdf')
display(Image(filename=str(excess_dir/'dlpfc_mean_excess_distribution.png')))
plt.close(fig)


In [ ]:
# Remaining datasets: exact excess histograms, then equal-experiment averages.
# Requires setup, cases, and support() above. Does not rerun alignments.
# Mean excess is conditional on violation. Cases with zero violations have no
# conditional distribution; their counts are reported separately, never set to zero.
from IPython.display import Image, display
OTHER=OUT/'other_dataset_excess_diagnostics';OTHER.mkdir(exist_ok=True)
reference=pd.read_csv(OUT/'violation_summary.csv',dtype={'case':str})
all_stats=[];group_summaries=[]
for dataset in ['MERFISH','Simulated overlap','Cropped windows','Deep-STARmap']:
    group=[c for c in cases if c['dataset']==dataset];histograms=[];group_stats=[]
    stem=dataset.lower().replace(' ','_').replace('-','_')
    folder=OTHER/stem;folder.mkdir(exist_ok=True)
    for c in group:
        xa,xb,_=coords(c);ii,jj,w,shape=support(c['path']);assert shape==(len(xa),len(xb))
        cutoff=c['recovery_cutoff_um'] if c['recovery_cutoff_um'] is not None else c['anchor_cutoff_um']
        role='recovery_numeric_reference' if c['recovery_cutoff_um'] is not None else 'anchor_pairwise'
        expected=reference[(reference.dataset==dataset)&(reference['case']==c['case'])&(reference.threshold_role==role)].iloc[0]
        ui,ir=np.unique(ii,return_inverse=True);uj,jr=np.unique(jj,return_inverse=True)
        da,db=cdist(xa[ui],xa[ui]),cdist(xb[uj],xb[uj])
        bins=np.arange(0,np.ceil(max(da.max(),db.max()))+2,1.)
        hist=np.zeros(len(bins)-1,dtype=np.int64);count=0;seen=0;sum_delta=0.;maximum=0.;m=len(w)
        for start in range(0,m,BLOCK):
            u=np.arange(start,min(start+BLOCK,m))
            for other in range(start,m,BLOCK):
                v=np.arange(other,min(other+BLOCK,m))
                delta=np.abs(da[ir[u,None],ir[v][None,:]]-db[jr[u,None],jr[v][None,:]])
                if other==start:delta=delta[np.triu_indices(len(u),1)]
                else:delta=delta.ravel()
                seen+=delta.size;bad=delta[delta>cutoff+ATOL];count+=len(bad)
                if len(bad):
                    hist+=np.histogram(bad-cutoff,bins=bins)[0]
                    sum_delta+=float(bad.sum());maximum=max(maximum,float(bad.max()))
        assert seen==m*(m-1)//2 and count==int(expected.violating_edge_pairs) and hist.sum()==count
        pd.DataFrame({'excess_bin_left_um':bins[:-1],'excess_bin_right_um':bins[1:],'count':hist}).to_csv(folder/f'{c["case"]}_excess_histogram.csv',index=False)
        mean=sum_delta/count if count else np.nan
        row=dict(dataset=dataset,case=c['case'],cutoff_um=cutoff,violating_combinations=count,total_combinations=seen,
                 ordinary_violation_rate=count/seen,mean_difference_violating_um=mean,
                 mean_excess_um=mean-cutoff,max_excess_um=maximum-cutoff if count else np.nan)
        all_stats.append(row);group_stats.append(row)
        if count:histograms.append(pd.Series(hist/count,index=bins[:-1]))
        print(dataset,c['case'],'completed',flush=True)
        pd.DataFrame(all_stats).to_csv(OTHER/'per_case_excess_summary.csv',index=False)
        del da,db,xa,xb;gc.collect()
    fig,ax=plt.subplots(figsize=(8,4.8),constrained_layout=True)
    if histograms:
        probability=pd.concat(histograms,axis=1).fillna(0).mean(axis=1).sort_index()
        assert np.isclose(probability.sum(),1.)
        probability=probability.loc[:probability[probability>0].index.max()]
        left=probability.index.to_numpy(dtype=float)
        mean_excess=float(np.mean([r['mean_excess_um'] for r in group_stats if r['violating_combinations']>0]))
        mean_hist=pd.DataFrame({'excess_bin_left_um':left,'excess_bin_right_um':left+1,'mean_violating_pairs_percent':100*probability.to_numpy()})
        ax.stairs(mean_hist.mean_violating_pairs_percent,np.r_[left,left[-1]+1],fill=True,color='#8DD3C7')
        ax.axvline(mean_excess,color='#436B68',linestyle='--',linewidth=1.4,label=f'Mean excess = {mean_excess:.2f} μm')
        ax.legend(frameon=False);ax.set_xlim(0,left[-1]+1)
    else:
        mean_excess=np.nan
        mean_hist=pd.DataFrame(columns=['excess_bin_left_um','excess_bin_right_um','mean_violating_pairs_percent'])
        ax.text(.5,.5,'No cutoff violations\nConditional excess distribution is undefined',ha='center',va='center',transform=ax.transAxes)
        ax.set_xlim(0,1);ax.set_xticks([]);ax.set_yticks([])
    mean_hist.to_csv(folder/f'{stem}_mean_excess_histogram.csv',index=False)
    group_summaries.append(dict(dataset=dataset,n_experiments=len(group),n_with_violations=len(histograms),
        n_without_violations=len(group)-len(histograms),mean_excess_um=mean_excess))
    ax.set_xlabel('Excess above full cutoff (μm)');ax.set_ylabel('Mean fraction of violating pairs (%)')
    ax.set_title(f'{dataset}: mean excess distribution (n={len(group)})')
    ax.set_ylim(bottom=0);ax.grid(axis='y',alpha=.2);ax.set_axisbelow(True)
    overall_rate=float(np.mean([r['ordinary_violation_rate'] for r in group_stats]))
    ax.text(.02,.30 if histograms else .75,f'Total violations\n{overall_rate:.2%}',transform=ax.transAxes,fontsize=11,fontweight='bold',color='#254B46',bbox=dict(facecolor='white',alpha=.8,edgecolor='none',pad=4))
    fig.savefig(folder/f'{stem}_mean_excess_distribution.png',dpi=200)
    fig.savefig(folder/f'{stem}_mean_excess_distribution.pdf')
    display(Image(filename=str(folder/f'{stem}_mean_excess_distribution.png')));plt.close(fig)
    pd.DataFrame(group_summaries).to_csv(OTHER/'dataset_mean_excess_summary.csv',index=False)
display(pd.DataFrame(group_summaries))
# Deep-STARmap μm follows the saved historical calibration convention; physical XY calibration is unverified.


In [ ]:
# Refresh all mean excess figures with the overall violation percentage.
# Overall percentage is the mean across experiments, not the sum of histogram bins.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display, Image
base=Path('/Users/salovjade/Library/CloudStorage/Dropbox/raftup_revision_828/all_exp/distortion_violation_audit/results_all')
jobs=[('DLPFC',base/'dlpfc_distance_diagnostics','dlpfc',
       pd.read_csv(base/'dlpfc_distance_diagnostics/dlpfc_violation_distance_summary.csv'))]
other=base/'other_dataset_excess_diagnostics'
remaining=pd.read_csv(other/'per_case_excess_summary.csv')
for dataset in ['MERFISH','Simulated overlap','Cropped windows','Deep-STARmap']:
    stem=dataset.lower().replace(' ','_').replace('-','_')
    jobs.append((dataset,other/stem,stem,remaining[remaining.dataset.eq(dataset)]))
for dataset,folder,stem,stats in jobs:
    h=pd.read_csv(folder/f'{stem}_mean_excess_histogram.csv')
    rate=stats.ordinary_violation_rate.mean()
    fig,ax=plt.subplots(figsize=(8,4.8),constrained_layout=True)
    if len(h):
        left=h.excess_bin_left_um.to_numpy()
        ax.stairs(h.mean_violating_pairs_percent,np.r_[left,left[-1]+1],fill=True,color='#8DD3C7')
        mean=stats.loc[stats.violating_combinations>0,'mean_excess_um'].mean()
        ax.axvline(mean,color='#436B68',linestyle='--',linewidth=1.4,label=f'Mean excess = {mean:.2f} μm')
        ax.legend(frameon=False);ax.set_xlim(0,left[-1]+1)
    else:
        ax.text(.5,.5,'No cutoff violations\nConditional excess distribution is undefined',ha='center',va='center',transform=ax.transAxes)
        ax.set_xlim(0,1);ax.set_xticks([]);ax.set_yticks([])
    ax.text(.02,.30 if len(h) else .75,f'Total violations\n{rate:.2%}',transform=ax.transAxes,
            fontsize=11,fontweight='bold',color='#254B46',
            bbox=dict(facecolor='white',alpha=.8,edgecolor='none',pad=4))
    ax.set_xlabel('Excess above full cutoff (μm)');ax.set_ylabel('Mean fraction of violating pairs (%)')
    ax.set_title(f'{dataset}: mean excess distribution (n={len(stats)})')
    ax.set_ylim(bottom=0);ax.grid(axis='y',alpha=.2);ax.set_axisbelow(True)
    fig.savefig(folder/f'{stem}_mean_excess_distribution.png',dpi=200)
    fig.savefig(folder/f'{stem}_mean_excess_distribution.pdf')
    display(Image(filename=str(folder/f'{stem}_mean_excess_distribution.png')));plt.close(fig)


## Combined violation prevalence and excess magnitude

One stacked bar per dataset combines the final excess-distribution plots. Each segment is the fraction of **all distinct positive-support correspondence-edge pairs** falling in an excess interval, averaged with equal weight per experiment. Bar lengths therefore reproduce the ordinary full-cutoff violation rates; these are not conditional percentages among violations. Zero-violation experiments contribute zero to every segment.

Excess is |dA − dB| minus the experiment’s full cutoff. Intervals align exactly with the saved 1-μm histogram bins. Full recovery thresholds are numerical pairwise references, not tests of anchor-residual feasibility. Deep-STARmap uses the saved historical μm convention. This cell runs independently from saved CSVs without rerunning alignment or pairwise enumeration.


In [ ]:
# Combined prevalence and excess severity; saved tables only, no audit rerun.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
base = Path('/Users/salovjade/Library/CloudStorage/Dropbox/raftup_revision_828/all_exp/distortion_violation_audit/results_all')
order = ['DLPFC', 'MERFISH', 'Simulated overlap', 'Cropped windows', 'Deep-STARmap']
bounds = np.array([0., 10., 25., 50., 100., np.inf])
bin_labels = ['0–<10', '10–<25', '25–<50', '50–<100', '≥100']
colors = ['#c6dbef', '#6baed6', '#3182bd', '#e89442', '#b74432']
dlpfc_dir = base / 'dlpfc_distance_diagnostics'
dlpfc = pd.read_csv(dlpfc_dir / 'dlpfc_violation_distance_summary.csv', dtype={'pair': str}).rename(columns={'pair': 'case'})
dlpfc['dataset'] = 'DLPFC'
other_dir = base / 'other_dataset_excess_diagnostics'
other = pd.read_csv(other_dir / 'per_case_excess_summary.csv', dtype={'case': str})
stats = pd.concat([dlpfc, other], ignore_index=True)
assert len(stats) == 51 and not stats.duplicated(['dataset', 'case']).any()
rows = []
for r in stats.itertuples():
    stem = r.dataset.lower().replace(' ', '_').replace('-', '_')
    folder = dlpfc_dir if r.dataset == 'DLPFC' else other_dir / stem
    hist = pd.read_csv(folder / f'{r.case}_excess_histogram.csv')
    left = hist.excess_bin_left_um.to_numpy()
    right = hist.excess_bin_right_um.to_numpy()
    count = hist['count'].to_numpy()
    assert r.total_combinations > 0 and count.sum() == r.violating_combinations
    np.testing.assert_allclose(right - left, 1.)
    # New boundaries coincide with saved 1-μm bin edges: no interpolation.
    segments = []
    for low, high, label in zip(bounds[:-1], bounds[1:], bin_labels):
        mask = (left >= low) & (left < high)
        assert np.all(right[mask] <= high)
        fraction = count[mask].sum() / r.total_combinations
        segments.append(fraction)
        rows.append(dict(dataset=r.dataset, case=r.case, excess_bin_um=label,
                         fraction_all_edge_pairs=fraction, percent_all_edge_pairs=100*fraction))
    np.testing.assert_allclose(sum(segments), r.ordinary_violation_rate, atol=1e-12)
per_case = pd.DataFrame(rows)
combined = per_case.groupby(['dataset', 'excess_bin_um']).percent_all_edge_pairs.mean().unstack().reindex(index=order, columns=bin_labels)
expected = pd.read_csv(base / 'dataset_mean_ordinary_violation.csv').set_index('dataset').reindex(order)
np.testing.assert_allclose(combined.sum(axis=1), expected.mean_violation_percent, atol=1e-10)
counts = stats.groupby('dataset').size().reindex(order)
per_case.to_csv(base / 'combined_violation_excess_per_case.csv', index=False)
table = combined.copy()
table['total_violation_percent'] = combined.sum(axis=1)
table['n_experiments'] = counts
table.to_csv(base / 'combined_violation_excess_summary.csv')
with plt.rc_context({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False}):
    fig, ax = plt.subplots(figsize=(11, 5.6))
    fig.subplots_adjust(left=.22, right=.97, top=.88, bottom=.30)
    y = np.arange(len(order)); cumulative = np.zeros(len(order))
    for label, color in zip(bin_labels, colors):
        values = combined[label].to_numpy()
        ax.barh(y, values, left=cumulative, height=.58, color=color,
                edgecolor='white', linewidth=.7, label=label)
        cumulative += values
    for i, total in enumerate(cumulative):
        ax.text(total + .14, i, f'{total:.2f}%', va='center', fontsize=11, fontweight='bold')
        if total == 0:
            ax.plot(0, i, '|', color='#64748b', markersize=16, clip_on=False)
    labels = [f'{name}  ({counts[name]} pairs)' for name in order]
    ax.set_yticks(y, labels); ax.invert_yaxis()
    ax.set_xlim(0, max(1., cumulative.max()) * 1.15)
    ax.set_xlabel('Mean fraction of all correspondence-edge pairs (%)', labelpad=10)
    ax.grid(axis='x', alpha=.2); ax.set_axisbelow(True)
    ax.tick_params(axis='y', length=0, pad=10)
    ax.legend(title='Excess above predefined cutoff (μm)', ncol=5, loc='upper center',
              bbox_to_anchor=(.5, -.25), frameon=False, borderaxespad=0,
              columnspacing=1.3, handlelength=1.4)
    fig.suptitle('Spatial-cutoff violations across experiments', fontsize=16, fontweight='bold', y=.98)
    for ext in ['png', 'pdf']:
        fig.savefig(base / f'combined_violation_excess.{ext}', dpi=240, bbox_inches='tight')
    plt.close(fig)
display(Image(filename=str(base / 'combined_violation_excess.png')))
display(table)
